In [18]:
import geopandas as gpd
import pandas as pd
import json
import time
import os
import requests

import sys

# use absolute path here
project_path = "/mnt/School/PhD/AI221/Project/"
sys.path.insert(0, project_path)

from src.data_extraction.utils.constants import data_path

In [4]:
ph_bounds = gpd.read_file(os.path.join(data_path,"ph_adm3_municities/PH_Adm3_MuniCities.shp.shp"))
ph_bounds = ph_bounds[ph_bounds["geo_level"] == "City"].to_crs(epsg=4326)
ph_bounds.head()

,adm1_psgc,adm2_psgc,adm3_psgc,adm3_en,geo_level,len_crs,area_crs,len_km,area_km2,geometry
4,100000000,102800000,102805000,City of Batac,City,66661,158252391,66,158.0,"POLYGON ((120.61242 18.10947, 120.612 18.10679..."
11,100000000,102800000,102812000,City of Laoag,City,53964,110146974,53,110.0,"POLYGON ((120.62081 18.22355, 120.62111 18.223..."
28,100000000,102900000,102906000,City of Candon,City,62247,77652664,62,77.0,"POLYGON ((120.46394 17.23489, 120.46412 17.234..."
56,100000000,102900000,102934000,City of Vigan,City,25067,24485368,25,24.0,"POLYGON ((120.37703 17.58121, 120.37817 17.581..."
70,100000000,103300000,103314000,City of San Fernando,City,54233,99006121,54,99.0,"POLYGON ((120.42381 16.6435, 120.42522 16.6417..."


In [15]:
# 3. Prepare the Output Directory
output_dir = "chirps_data"
os.makedirs(
    os.path.join(data_path, "rainfall", output_dir),
    exist_ok=True
)

In [19]:
# --- 1. Custom API Caller (Now with Monthly Aggregation) ---
def get_chirps_data(points_list, start_date, end_date, outfile):
    base_url = "https://climateserv.servirglobal.net/api/"
    
    geom_json = json.dumps({
        "type": "Polygon", 
        "coordinates": [points_list],
        "properties": {}
    })
    
    payload = {
        "datatype": 0,               # CHIRPS Global Daily
        "begintime": start_date,
        "endtime": end_date,
        "intervaltype": 0,
        "operationtype": 5,          # 5 = Average (Zonal Mean)
        "dateType_Category": "default",
        "isZip_CurrentDataType": "False",
        "geometry": geom_json
    }

    try:
        submit_res = requests.post(base_url + "submitDataRequest/", data=payload)
        submit_res.raise_for_status()
        job_id = submit_res.json()[0]
        
        while True:
            prog_res = requests.get(base_url + "getDataRequestProgress/", params={"id": job_id})
            progress = prog_res.json()[0]
            
            if progress == 100.0:
                break
            if progress == -1.0:
                print("  ✗ Server returned a computation error (-1).")
                return False
            time.sleep(1)
            
        data_res = requests.get(base_url + "getDataFromRequest/", params={"id": job_id})
        raw_data = data_res.json()
        
        records = []
        for item in raw_data.get('data', []):
            date_val = item.get('date', '')
            # Handle potential nulls safely
            try:
                avg_val = float(item.get('value', {}).get('avg', 0.0))
            except (TypeError, ValueError):
                avg_val = 0.0
                
            records.append({'date': date_val, 'avg': avg_val})
            
        # --- THE FIX: Pandas Monthly Aggregation ---
        df = pd.DataFrame(records)
        df['date'] = pd.to_datetime(df['date'])
        
        # Group by month (Month Start) and sum the daily rainfall
        monthly_df = df.set_index('date').resample('MS').sum().reset_index()
        
        # Format the date nicely to YYYY-MM (e.g., "2022-01")
        monthly_df['date'] = monthly_df['date'].dt.strftime('%Y-%m')
        
        # Save straight to CSV, perfectly formatted
        monthly_df.to_csv(outfile, index=False)
        return True

    except Exception as e:
        print(f"  ✗ API Error: {e}")
        return False

In [20]:
for index, row in ph_bounds.iterrows():
    muni_name = str(row['adm3_psgc']).replace('/', '_')
    outfile = f"{output_dir}/{muni_name}_chirps_2022_2025.csv"
    
    if os.path.exists(outfile):
        print(f"Skipping {muni_name} -> Already exists.")
        continue
        
    print(f"Requesting data for: {muni_name}...")
    geom = row['geometry']
    
    # Safe Geometry Extraction (with 1km simplification)
    try:
        geom = geom.simplify(tolerance=0.01, preserve_topology=True)
        if geom.geom_type == 'Polygon':
            points = [[x, y] for x, y in geom.exterior.coords]
        elif geom.geom_type == 'MultiPolygon':
            largest_poly = max(geom.geoms, key=lambda p: p.area)
            points = [[x, y] for x, y in largest_poly.exterior.coords]
        else:
            continue
            
        # Force closure just in case
        if points[0] != points[-1]:
            points.append(points[0])
            
    except Exception as e:
        print(f"  ✗ Geometry error: {e}")
        continue
        
    # Call our custom function
    success = get_chirps_data(points, "01/01/2022", "12/31/2025", outfile)
    
    if success:
        print(f"  ✓ Saved to CSV")
        
    time.sleep(1.5) # Politeness delay

print("\nExtraction complete!")

Requesting data for: 102805000...
  ✓ Saved to CSV
Requesting data for: 102812000...
  ✓ Saved to CSV
Requesting data for: 102906000...
  ✓ Saved to CSV
Requesting data for: 102934000...
  ✓ Saved to CSV
Requesting data for: 103314000...
  ✓ Saved to CSV
Requesting data for: 105503000...
  ✓ Saved to CSV
Requesting data for: 105518000...
  ✓ Saved to CSV
Requesting data for: 105532000...
  ✓ Saved to CSV
Requesting data for: 105546000...
  ✓ Saved to CSV
Requesting data for: 201529000...
  ✓ Saved to CSV
Requesting data for: 203108000...
  ✓ Saved to CSV
Requesting data for: 203114000...
  ✓ Saved to CSV
Requesting data for: 203135000...
  ✓ Saved to CSV
Requesting data for: 300803000...
  ✓ Saved to CSV
Requesting data for: 301403000...
  ✓ Saved to CSV
Requesting data for: 301410000...
  ✓ Saved to CSV
Requesting data for: 301412000...
  ✓ Saved to CSV
Requesting data for: 301420000...
  ✓ Saved to CSV
Requesting data for: 304903000...
  ✓ Saved to CSV
Requesting data for: 304908000.

In [27]:
# Special run for 704604000 since it failed above
for index, row in ph_bounds.iterrows():
    if row['adm3_psgc'] != 704604000:
        continue
        
    muni_name = str(row['adm3_psgc']).replace('/', '_')
    outfile = f"{output_dir}/{muni_name}_chirps_2022_2025.csv"
    
    if os.path.exists(outfile):
        print(f"Skipping {muni_name} -> Already exists.")
        continue
        
    print(f"Requesting data for: {muni_name}...")
    geom = row['geometry']
    
    # Safe Geometry Extraction (with 1km simplification)
    try:
        geom = geom.simplify(tolerance=0.01, preserve_topology=True)
        if geom.geom_type == 'Polygon':
            points = [[x, y] for x, y in geom.exterior.coords]
        elif geom.geom_type == 'MultiPolygon':
            largest_poly = max(geom.geoms, key=lambda p: p.area)
            points = [[x, y] for x, y in largest_poly.exterior.coords]
        else:
            continue
            
        # Force closure just in case
        if points[0] != points[-1]:
            points.append(points[0])
            
    except Exception as e:
        print(f"  ✗ Geometry error: {e}")
        continue
        
    # Call our custom function
    success = get_chirps_data(points, "01/01/2022", "12/31/2025", outfile)
    
    if success:
        print(f"  ✓ Saved to CSV")
        
    time.sleep(1.5) # Politeness delay

print("\nExtraction complete!")

Requesting data for: 704604000...
  ✓ Saved to CSV

Extraction complete!
